In [38]:
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import tokenizers

In [39]:
from datasets import load_dataset_builder

Building an English to Luganda Neural Machine Translation model using an encoder-decoder transformer built from scratch

In [40]:
build = load_dataset_builder("kambale/luganda-english-parallel-corpus")

In [41]:
data = load_dataset("kambale/luganda-english-parallel-corpus")

In [42]:
data

DatasetDict({
    train: Dataset({
        features: ['english', 'luganda'],
        num_rows: 50012
    })
})

Just 50,012 examples and only one split: train. Need to break this down to include test and validation splits - 5% each

In [43]:
split = data['train'].train_test_split(train_size=0.95, seed=42)
train_valid, test_data = split['train'],split['test']

In [44]:
split2 = train_valid.train_test_split(train_size=0.95, seed=42)
train_data, valid_data = split2['train'], split2['test']

Done with splitting the data. Now need to visualise the data....

In [45]:
for i in range(5):
    print(train_data[i])

{'english': 'The cabinet approved the construction of the hydropower station on Murchison falls.', 'luganda': "Akakiiko kaayisizza okuzimba kw'ebbibiro ly'amasannyalaze ku biyiriro bya Murchison."}
{'english': 'Many farmers lost their crops in the flood.', 'luganda': 'Abalimi bangi baafiirwa ebirime byabwe mu mataba.'}
{'english': 'She climbed the tree to pick mangoes.', 'luganda': 'Yalinnya omuti okufuna emiyembe.'}
{'english': 'The government has released money to support the footballers in international competitions.', 'luganda': "Gavumenti ewaddeyo ssente ez'okuyamba abazannyi b'omupiira mu mpaka ez'ensi yonna."}
{'english': 'The veterans agreed to stay united and harmonize with each other.', 'luganda': "Abaazirwanako bakiriziganyizza okusigala ekitole n'okukwatagana na buli omu.."}


Looks like good old Luganda to me. Next up, need to train a custom tokenizer for this, then batch up my data and get it ready for ml training and inference.

Notes:
- Going to use the same tokenizer for both languages as they are similar in nature. Both separated by spaces and both minimal on use of accents

In [46]:
# need a function that will yield sentences each time its iterated over without having everything loaded into ram
# a generator
def train_eng_lug():
    for pair in train_data:
        yield pair['english']
        yield pair['luganda']

In [47]:
max_length = 512 # luganda is quite verbose
vocab_size = 15000

tokenizer_model = tokenizers.models.BPE(unk_token="<unk>") # using byte pair encoding and not specifying the unk_token as I intend on using byte level pretokenization
tokenizer = tokenizers.Tokenizer(tokenizer_model)
tokenizer.enable_padding(pad_id=0, pad_token="<pad>") # enabling padding for when i call tokenizer.encode_batch on a batch with sentences of varying token lengths
tokenizer.enable_truncation(max_length=max_length) # don't want my model to train forever so limiting the max_length of tokens from sequences to 512
tokenizer.normalizer = tokenizers.normalizers.Lowercase() # just going to lowercase all tokens
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace() # going with a byte level tokenizer here to ensure there isn't any unknown tokens
tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>","<unk>","<s>","</s"]
)
tokenizer.train_from_iterator(train_eng_lug(), tokenizer_trainer)

In [48]:
# load the device
device = torch.device("mps" if torch.mps.is_available() else "cpu")

Now need to work on how the data is collected and fed to the model:
- Building a nmt transformer so essentially, for training - actual english sentences going to the encoder and luganda sentences shifted one timestep backward going to the decoder (teacher forced training). then the normal luganda sentences ending with an eos token as the targets.
- need the attention masks as well computed by the tokenizer as these will later be used in the transformer model to generate and produce the key padding masks needed to mask the padding tokens when computing self attention

In [49]:
from collections import namedtuple

In [50]:
# using named tuples as there's lots of data variables and instead of remembering them by index, I prefer to remember them by name
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"] # named tuple fields
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device): # adding a custom to function 
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device), self.tgt_token_ids.to(device),
                      self.tgt_mask.to(device))

In [51]:
#  till this point haven't converted my train, validation, and test datasets to actual tokens that get fed to the model
# need to do that now
def collate_fn(batch):
    # what we have at this point in terms of the batch coming in is a tuple of dictionaries
    # with each dictionary containing a key: english and luganda
    src_texts = [pair['english'] for pair in batch] # getting the source english texts from the batch
    tgt_texts = [f"<s> {pair['luganda']} </s>" for pair in batch] # adding the sos and eos tokens to the target texts as well
    # now getting the encodings
    src_encodings = tokenizer.encode_batch(src_texts) # returned encodings of shape: BxL
    tgt_encodings = tokenizer.encode_batch(tgt_texts) # returned encodings of shape: BxL
    # need to extract the encodings and mask from the returned objects - remember both are of shape BxL
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    # now masks
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])

    # now need to input these into the created class
    inputs = NmtPair(src_token_ids, src_mask, tgt_token_ids[:,:-1], tgt_mask[:,:-1]) 
    # as this is teacher forced training, we exclude the last eos token from the model's inputs
    labels = tgt_token_ids[:,1:] # the labels will constitute the target token ids with the sos token excluded - classic teacher forced training
    return inputs, labels

In [52]:
batch_size = 32 # keeping this fairly low cuz of my device - 24GB UNIFIED memory base m4 mac mini
train_loader = DataLoader(train_data, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=batch_size, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=batch_size, collate_fn=collate_fn)

testing out whats just been built:

In [53]:
d, l = next(iter(train_loader))

In [54]:
d.src_token_ids.shape, d.src_mask.shape, d.tgt_token_ids.shape, d.tgt_mask.shape

(torch.Size([32, 17]),
 torch.Size([32, 17]),
 torch.Size([32, 23]),
 torch.Size([32, 23]))

In [55]:
l.shape

torch.Size([32, 23])

Works just fine......

**Model Building**
- Transformer from scratch...

First, we build the positional encoding custom pytorch layer...

In [56]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length,embed_dim)*0.02) # this is the learnable positional encoding matrix of this layer
        self.dropout = nn.Dropout(dropout)
    def forward(self, X):
        # implementing how this layer should work
        # essentially - get the original embeddings and then add the positional encodings
        # slice the positional embeddings learnable weight matrix by the sequence length of the input matrix (assumed to be of shape: B,L)
        return self.dropout(X + self.pos_embed[:X.size(1)])

Next layer thats needed from scratch is the multihead attention layer 
- will support and enable multihead attention for the encoder and masked multihead attention for the decoder as well as cross attention for the decoder...

In [57]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        # define the learnable projection layers - key intuition is that these teach each attention head the right questions to ask
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        # define the output projection as well - key idea is it reinforces the learned contextual embeddings after being concatenated from the heads
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        # define dropout
        self.dropout = nn.Dropout(dropout)
    
    # now split the heads - to get the attention heads
    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1,2) # returning: BxhxLxd
    
    # implementing self attention across the different heads
    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        # project and split
        q = self.split_heads(self.q_proj(query)) # input query: B,L,embed_dim | output q: B,h,Lq,dq
        k = self.split_heads(self.k_proj(key)) # input key: B,L,embed_dim | output k: B,h,Lk,dk
        v = self.split_heads(self.v_proj(value)) # input value: B,L, embed_dim | output v: B,h,Lv,dv

        # rule of thumb is that dq = dk. now, for each token in the query, I compute the resultant dot product score between it and every token in the key
        # this is the scaled attention computation - because we split into heads earlier, this is happening across all heads in parallel
        scores = q @ k.transpose(2,3) / self.d**0.5

        # need to mask out some of these attention scores as depending on the computational context, not all of them should contribute to the final output
        # if the nature of the task is causal - then only the scores from that current token to the previous ones should contribute
        # scores from padding tokens should also be nullified
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf) # using the mask, fill out the unnecesary tokens with -torch.inf later zeroed out by softmax
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2) # transforming original mask from B,L shape to B,1,1,L
            scores = scores.masked_fill(mask, -torch.inf)
        
        # now compute the softmax rep of the weights
        weights = scores.softmax(dim=-1) # shape is still: B,h,Lq,Lk
        # use the weights to get the weighted contribution of each value from the key array and then get final contextual embeddings
        Z = self.dropout(weights) @ v # output: B,h,Lq,dv
        # now need to return original shape
        Z = Z.transpose(1,2)
        Z = Z.reshape(Z.size(0),Z.size(1), self.h*self.d)
        return (self.out_proj(Z), weights)    


and thats the multihead attention layer...... used in both the encoder and decoder which are now implemented below...

In [58]:
# a single encoder layer has...
# multiattention head, and a feedforward net
# check the attention is all you need paper for the original implementation
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        attn, _ = self.self_attn(src,src,src,attn_mask=src_mask,key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(\
                        self.linear2(\
                            self.dropout(\
                                self.linear1(Z).relu()
                                )))
        return self.norm2(Z + ff)

In [59]:
# now build the basic decoder layer
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.multihead_attn = MultiHeadAttention(d_model, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        # masked self attention
        attn1, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))
        # cross attention
        # the masked self attention output is the query, the encoder output (memory) is the key and value
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=memory_mask, key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        ff = self.dropout(\
            self.linear2(\
                self.dropout(\
                    self.linear1(Z).relu()
                    )
                )
            )
        return self.norm3(Z + ff)

Now, need to use these encoder and decoder layers inside encoder and decoder blocks that call these layers N times.......

In [60]:
from copy import deepcopy
# need to maintain a clean syntax whilst copying the encoder and decoder layers n times

In [61]:
class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(encoder_layer) for _ in range(num_layers)])
        self.norm = norm
    
    def forward(self, src, mask=None, src_key_padding_mask=None):
        Z = src
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [62]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(decoder_layer) for _ in range(num_layers)])
        self.norm = norm
    
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        Z = tgt 
        for layer in self.layers:
            Z = layer(Z, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [63]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm1 = nn.LayerNorm(d_model)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, norm1)

        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        norm2 = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, norm2)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        memory = self.encoder(src, src_mask, src_key_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask, tgt_key_padding_mask, memory_key_padding_mask)
        return output

Integrating the above built models into the model:

In [64]:
class EnglishLugandaTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0, num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEncoding(max_length, embed_dim, dropout)
        self.transformer = Transformer(embed_dim, num_heads, num_layers, num_layers, dropout=dropout)
        self.output= nn.Linear(embed_dim, vocab_size)
    
    def forward(self, pair):
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))

        # generate the key padding token masks
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()

        # now build the full attention masks - causal one for the target and nothing for the source
        size = [pair.tgt_token_ids.size(1)]*2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)

        # convert to a causal mask
        causal_mask = torch.triu(full_mask, diagonal=1)

        out_decoder = self.transformer(src_embeds, tgt_embeds, src_key_padding_mask=src_pad_mask,\
                                       memory_key_padding_mask=src_pad_mask, tgt_mask=causal_mask,\
                                        tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0,2,1)

Training the model.....

In [65]:
torch.manual_seed(42)

In [66]:
vocab_size

15000

In [71]:
eng_lug_model = EnglishLugandaTransformer(vocab_size, max_length, embed_dim=128, num_heads=4, num_layers=2, dropout=0.1).to(device)

In [72]:
from utils.early_stopping import EarlyStopping

In [73]:
xentropy = nn.CrossEntropyLoss(ignore_index=0, reduction='sum')
optimizer = torch.optim.NAdam(eng_lug_model.parameters())
early_stopper = EarlyStopping(patience=20, checkpoint_path='eng_lug.pt', restore_best_weights=True, verbose=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',patience=5,factor=0.9)

In [74]:
n_epochs = 500

train_loss = [0]*n_epochs
val_loss = [0]*n_epochs

for epoch in range(n_epochs):
    eng_lug_model.train()
    # iterate through the training data
    for inputs,labels in train_loader:
        inputs,labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = eng_lug_model(inputs)
        loss =  xentropy(out, labels)
        loss.backward()
        optimizer.step()
        train_loss[epoch] += loss.item()
    train_loss[epoch] /= len(train_loader.dataset)

    # eval on the val set
    eng_lug_model.eval()
    with torch.no_grad():
        for inputs, labels in valid_loader:
            inputs,labels = inputs.to(device),labels.to(device)
            out = eng_lug_model(inputs)
            loss = xentropy(out, labels)
            val_loss[epoch] += loss.item()
        val_loss[epoch] /= len(valid_loader.dataset)

        scheduler.step(val_loss[epoch])
        print(f'Epoch: {epoch+1}| Train loss: {train_loss[epoch]:.4f}| Val loss: {val_loss[epoch]:.4f}')
        early_stopper(val_loss[epoch], eng_lug_model, optimizer, epoch)
        if early_stopper.should_stop:
            print(f"Stopping at epoch: {epoch+1}")
            break

Epoch: 1| Train loss: 64.2592| Val loss: 56.8551
Metric improved to 56.8551. Checkpoint saved at epoch 0
Epoch: 2| Train loss: 52.2583| Val loss: 50.9599
Metric improved to 50.9599. Checkpoint saved at epoch 1
Epoch: 3| Train loss: 46.9365| Val loss: 48.1354
Metric improved to 48.1354. Checkpoint saved at epoch 2
Epoch: 4| Train loss: 43.6057| Val loss: 45.6935
Metric improved to 45.6935. Checkpoint saved at epoch 3
Epoch: 5| Train loss: 41.0489| Val loss: 43.9169
Metric improved to 43.9169. Checkpoint saved at epoch 4
Epoch: 6| Train loss: 38.9551| Val loss: 42.2934
Metric improved to 42.2934. Checkpoint saved at epoch 5
Epoch: 7| Train loss: 37.2636| Val loss: 41.2212
Metric improved to 41.2212. Checkpoint saved at epoch 6
Epoch: 8| Train loss: 35.7722| Val loss: 40.0017
Metric improved to 40.0017. Checkpoint saved at epoch 7
Epoch: 9| Train loss: 34.4982| Val loss: 38.6395
Metric improved to 38.6395. Checkpoint saved at epoch 8
Epoch: 10| Train loss: 33.3154| Val loss: 37.5792
Metri

KeyboardInterrupt: 

stopped cuz of how long it was taking to train,....that should be enough

remember the input text is processed by the collate function to return the input tokens, their masks and labels... its this that then gets passed to th emodel

In [75]:
in_, _ = collate_fn([{"english": "The Government", "luganda": ""}])
# collate fn expects a list of these dictionaries - passed the english text that should go the encoder and to the decoder passed nothing so will get <s> </s>
with torch.no_grad():
    out_ = eng_lug_model(in_.to(device))

In [76]:
out_.shape

torch.Size([1, 15000, 2])

In [77]:
out_2 = out_.argmax(dim=1)

In [78]:
out_2.shape

torch.Size([1, 2])

In [81]:
otok = tokenizer.id_to_token(out_2[0,1])

In [82]:
otok

'>'

In [83]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = collate_fn([{"english": src_text,
                                    "luganda": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [96]:
eng_lug_model.eval()
translate(eng_lug_model, "happy new year")

' buli mwaka ssetendekero abapya . </s'